# Mi optimización bayesiana de origen

Documento y puedo volver a ejecutar la BO ORIGINAL_LAG2 que seleccionó los HP usados en mi entrega. No la confundo con la BO de Sin cierres posterior, que descarté.

## 1. Carga — Python

In [ ]:
# Ubico el crudo y preparo mi carpeta de trabajo sin modificar el dataset.
from pathlib import Path
import hashlib, json, os, urllib.request
carpeta = Path.cwd()
datos = carpeta / 'datos'
datos.mkdir(exist_ok=True)
crudo = Path(os.environ.get('DMEYF_DATOS', str(datos / 'competencia_01_crudo.csv')))
if not crudo.is_file():
    crudo.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/competencia_01_crudo.csv', crudo)
with crudo.open('rb') as f:
    md5 = hashlib.file_digest(f, 'md5').hexdigest()
assert md5 == 'b6f759764c5d04cbf068548517ae1249', 'El crudo no coincide con mi corrida.'
(carpeta / 'RUTA_DATOS.txt').write_text(str(crudo.resolve()))
print('Dataset verificado:', crudo)


## 2. Optimización — R

Cambio al kernel R. Uso marzo–abril para entrenar, junio para evaluar y K11500 como objetivo de esta BO. Mantengo 4 puntos iniciales y 8 propuestas por EI.

In [ ]:
Sys.setenv(DMEYF_DATOS=readLines('RUTA_DATOS.txt',warn=FALSE)[1])
# Fijo mi entrega y dejo apagadas las variantes que no elegí.
PARAM <- list(
 semilla=100613L,
 semillas=c(100613L,100267L,101611L,102481L,103651L,104729L,
            105019L,105613L,106013L,106619L),
 threads=6L,train=c(202103L,202104L),
 train_final=c(202103L,202104L,202105L,202106L),
 valid=202106L,future=202108L,corte=11500L,orden_lags=2L,
 incluir_id_mes=TRUE,bo_evaluaciones=12L,bo_iniciales=4L,
 bounds=list(iter=c(512L,4096L),leaves=c(8L,256L),logh=c(-3,1)),
 top_configuraciones=1L,rank_campos=character()
)
SW <- list(
 reproducir_final=TRUE,bo_origen=FALSE,
 control=FALSE,rankings=FALSE,artificiales_original=FALSE,
 artificiales_rank=FALSE,rank_montos=FALSE,target_baja2=FALSE,
 tarjetas=FALSE,solo_visa=FALSE,solo_master=FALSE,
 vinculacion=FALSE,sin_cierres_historico=FALSE,
 rank_zero_tend3=FALSE,bo_sin_cierres_nueva=FALSE,
 diagnosticos_ens5=FALSE,cortes_historicos=FALSE
)
CORTE_FINAL <- 10375L
SHA_FINAL <- '7715cbb9568462c2a1f9b0534528fbce3f77c54cd43a255c59476dc9a37b5caa'
CARPETA <- normalizePath(Sys.getenv('DMEYF_ENTREGA',unset=getwd()),mustWork=TRUE)
if(!file.exists(file.path(CARPETA,'codigo','motor.R')))
 stop('Debo ejecutar desde la carpeta de entrega o definir DMEYF_ENTREGA.')
RAIZ_NUEVA <- Sys.getenv('DMEYF_SALIDA',unset=file.path(CARPETA,'salida'))
RAW_NUEVO <- Sys.getenv('DMEYF_DATOS',unset=file.path(CARPETA,'datos','competencia_01_crudo.csv'))
if(!file.exists(RAW_NUEVO)) stop('Debo indicar DMEYF_DATOS con el dataset crudo.')
if(unname(tools::md5sum(RAW_NUEVO))!='b6f759764c5d04cbf068548517ae1249')
 stop('El crudo no coincide con el dataset de la corrida final.')
Sys.setenv(DMEYF_RAIZ=RAIZ_NUEVA,DMEYF_DATOS=RAW_NUEVO)
if(dir.exists('/home/ds/.local/lib/R/site-library'))
 .libPaths(unique(c('/home/ds/.local/lib/R/site-library',.libPaths())))
paquetes <- c('data.table','lightgbm','digest','jsonlite','filelock')
faltan <- paquetes[!vapply(paquetes,requireNamespace,logical(1),quietly=TRUE)]
if(length(faltan)) stop('Debo instalar estos paquetes: ',paste(faltan,collapse=', '))
library(lightgbm)
stopifnot(as.character(getRversion())=='4.6.1',
 as.character(packageVersion('lightgbm'))=='4.7.0.99',
 as.character(packageVersion('data.table'))=='1.18.6.1')
source(file.path(CARPETA,'codigo','motor.R'),local=.GlobalEnv)

SW$reproducir_final <- FALSE
SW$bo_origen <- TRUE


In [ ]:
# Reconstruyo el target, corrijo mayo y calculo la historia calendario exacta.
D <- cargar_base()
REP <- preparar_features(D,FALSE)
stopifnot(REP$id=='c9527308c55d2d52c924dc64ce5594db3d240bce96c8a548c6f48a06239c71c8')
rm(D)
gc(full=TRUE)
# Dejo anotado: Recupero la procedencia de los HP sin hacer pasar una BO nueva por la original.
ORIGEN <- readRDS(file.path(CARPETA,'procedencia_bo',
 '52fc1b5d908fb495ce06d1ed3038defbaf9a2609d2ea0bc80904d2ba8b58c6ed',
 'TERMINADO.rds'))
HP_FINAL <- ORIGEN$valor$optimos[[1L]]
stopifnot(HP_FINAL$num_iterations==1907L,HP_FINAL$num_leaves==77L,
 abs(HP_FINAL$min_sum_hessian_in_leaf-2.83699883991516)<1e-12)
REC_CONTROL <- list(nombre='ORIGINAL_LAG2',rep=REP,estrategia='COMPLETO',
 umbral=NULL,seleccion=NULL,excluir=character(),planes=NULL)
# Dejo anotado: Si activo la BO de origen, la ejecuto en otra raíz sin reutilizar su resultado.
if(SW$bo_origen) {
 for(paquete in c('mlrMBO','mlr','ParamHelpers','smoof','DiceKriging'))
  if(!requireNamespace(paquete,quietly=TRUE))stop('Debo instalar ',paquete)
 b <- bayes(REC_CONTROL)
 atomic(b,file.path(RAIZ,'BO_ORIGEN_REEJECUTADA.rds'))
}


In [ ]:
# Comparo el resultado con la procedencia que guardé.
print(ORIGEN$valor$historial)
print(HP_FINAL)


Para generar el CSV final vuelvo a la notebook de entrega con sus switches predeterminados. El historial guardado documenta los HP efectivamente usados; una reejecución de BO no sustituye silenciosamente esa selección.